# Day 14 — Logging, Config, Output & Manifest

Pipeline driven by `config/config.yaml`. Runs are idempotent — second run skips already-processed files.

In [27]:
# Parameters
CONFIG_PATH = "../config/config.yaml"

In [28]:
import sys, time
from pathlib import Path
sys.path.insert(0, str(Path("../src").resolve()))

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Task 3 — Load config.yaml

In [29]:
from pipeline import load_config

cfg = load_config(CONFIG_PATH)
print("Config loaded:")
import json
print(json.dumps(cfg, indent=2))

Config loaded:
{
  "paths": {
    "input_folder": "../data",
    "output_folder": "../data/output",
    "log_folder": "../logs",
    "manifest_file": "../data/output/manifest.json",
    "bad_records_file": "../data/output/bad_records.csv",
    "column_mapping": "../config/column_mapping.json"
  },
  "files": [
    "orders.csv",
    "order_items.csv",
    "customers.csv"
  ],
  "thresholds": {
    "rejection_rate": 0.2
  },
  "pipeline": {
    "float_cols": [
      "price",
      "freight_value"
    ],
    "datetime_cols": [
      "shipping_limit_at"
    ],
    "required_fields": {
      "orders.csv": [
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_estimated_delivery_date"
      ],
      "order_items.csv": [
        "order_id",
        "product_id",
        "seller_id",
        "price",
        "freight_value",
        "shipping_limit_date"
      ],
      "customers.csv": [
        "customer_id",
        "customer_u

## Tasks 1 & 2 — Logging with timestamps

In [30]:
from pipeline import setup_logging

logger = setup_logging(cfg["paths"]["log_folder"])
logger.info("Pipeline initialised from config.yaml")

2026-11-25 14:35:46 [INFO] Logging started — log file: ../logs\pipeline_20261125_143546.log
2026-11-25 14:35:46 [INFO] Pipeline initialised from config.yaml


## Tasks 6 & 7 & 10 — Full pipeline: manifest check, validate, output, summary

In [31]:
import os
from pipeline import (
    load_manifest, save_manifest, is_already_processed,
    write_output_csv, write_output_json, write_run_summary,
)
from validator13 import validate_file
from schema import SchemaError, FileFormatError
from reader import read_csv
from validator import load_column_mapping, apply_column_mapping
from transformer import convert_types

start_time  = time.time()
paths       = cfg["paths"]
manifest    = load_manifest(paths["manifest_file"])
mapping     = load_column_mapping(paths["column_mapping"])
summaries   = []
skipped     = []

for filename in cfg["files"]:
    # Task 6 — skip already processed files
    if is_already_processed(filename, manifest):
        logger.info(f"SKIP {filename} — already processed on {manifest[filename]}")
        skipped.append(filename)
        continue

    filepath  = os.path.join(paths["input_folder"], filename)
    required  = cfg["pipeline"]["required_fields"].get(filename, [])
    bad_path  = paths["bad_records_file"]

    try:
        logger.info(f"Validating {filename}...")
        summary = validate_file(
            filepath, required,
            bad_records_path=bad_path,
            rejection_threshold=cfg["thresholds"]["rejection_rate"],
        )
        summaries.append(summary)
        logger.info(f"{filename}: {summary['valid']} valid, {summary['rejected']} rejected")

        # Task 4 — write clean output as CSV and JSON
        rows = read_csv(filepath)
        rows_mapped = apply_column_mapping(rows, mapping)
        rows_typed  = convert_types(
            rows_mapped,
            cfg["pipeline"]["float_cols"],
            cfg["pipeline"]["datetime_cols"],
        )
        stem = Path(filename).stem
        out_csv  = os.path.join(paths["output_folder"], f"{stem}_clean.csv")
        out_json = os.path.join(paths["output_folder"], f"{stem}_clean.json")
        write_output_csv(rows_typed, out_csv)
        write_output_json(rows_typed, out_json)
        logger.info(f"Output written: {out_csv}, {out_json}")

        # Task 6 — record in manifest
        from datetime import datetime
        manifest[filename] = datetime.now().isoformat()

    except SchemaError as e:
        logger.error(f"SCHEMA ERROR {filename}: {e}")
    except FileFormatError as e:
        logger.warning(f"FILE ERROR {filename}: {e}")

save_manifest(manifest, paths["manifest_file"])

# Task 5 — write run summary
summary_path = os.path.join(paths["output_folder"], "run_summary.json")
write_run_summary(summaries, start_time, summary_path)

# Task 9 — print summary
print(f"\n{'='*50}")
print(f"Files processed : {len(summaries)}")
print(f"Files skipped   : {len(skipped)} {skipped}")
for s in summaries:
    print(f"  {s['file']:25s} total={s['total']:4d}  valid={s['valid']:4d}  rejected={s['rejected']:3d}  rate={s['rejection_rate']:.1%}")
print(f"Run summary     : {summary_path}")
print(f"Manifest        : {paths['manifest_file']}")

2026-11-25 14:35:46 [INFO] SKIP orders.csv — already processed on 2026-11-25T10:24:10.305556


2026-11-25 14:35:46 [INFO] SKIP order_items.csv — already processed on 2026-11-25T10:24:10.343025
2026-11-25 14:35:46 [INFO] SKIP customers.csv — already processed on 2026-11-25T10:24:10.362590



Files processed : 0
Files skipped   : 3 ['orders.csv', 'order_items.csv', 'customers.csv']
Run summary     : ../data/output\run_summary.json
Manifest        : ../data/output/manifest.json


## Task 8 — Lint with ruff

In [32]:
!python -m ruff check ../src/ --output-format=concise

All checks passed!


## Task 10 — Second run: shows all files skipped

In [ ]:
print("Re-running pipeline (second run)...")
manifest2  = load_manifest(paths["manifest_file"])
skipped2   = []

for filename in cfg["files"]:
    if is_already_processed(filename, manifest2):
        logger.info(f"SKIP {filename} — already processed on {manifest2[filename]}")
        skipped2.append(filename)
    else:
        logger.info(f"Would process {filename}")

print(f"\nSecond run result: {len(skipped2)}/{len(cfg['files'])} files skipped")
for f in skipped2:
    print(f"  SKIPPED: {f} (processed on {manifest2[f]})")


2026-11-25 14:35:46 [INFO] SKIP orders.csv — already processed on 2026-11-25T10:24:10.305556
2026-11-25 14:35:46 [INFO] SKIP order_items.csv — already processed on 2026-11-25T10:24:10.343025


2026-11-25 14:35:46 [INFO] SKIP customers.csv — already processed on 2026-11-25T10:24:10.362590


Re-running pipeline (second run)...

Second run result: 3/3 files skipped
  SKIPPED: orders.csv (processed on 2026-11-25T10:24:10.305556)
  SKIPPED: order_items.csv (processed on 2026-11-25T10:24:10.343025)
  SKIPPED: customers.csv (processed on 2026-11-25T10:24:10.362590)
hi
